In [2]:
from google.colab import drive
# Attempt to unmount first to ensure a clean state
drive.flush_and_unmount()

# Remove any lingering files in the mountpoint directory
!rm -rf /content/drive/*

drive.mount('/content/drive', force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [6]:
import os

image_directory = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/train_images"

# Check if the directory exists
if not os.path.exists(image_directory):
    print(f"Error: Directory not found at {image_directory}")
else:
    # List all files in the directory
    all_files = os.listdir(image_directory)

    # Filter for image files (you can add more extensions if needed)
    image_extensions = ('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tif')
    photo_count = 0
    for filename in all_files:
        if filename.lower().endswith(image_extensions):
            photo_count += 1

    print(f"Number of photos in '{image_directory}': {photo_count}")

Number of photos in '/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/train_images': 849


In [7]:
import sys
!{sys.executable} -m pip install geopandas

In [8]:
import geopandas as gpd

geojason_file_path = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/train_labels/xView_train.geojson"

# Load the GeoJSON file into a GeoDataFrame
try:
    gdf = gpd.read_file(geojason_file_path)
    print(f"Successfully loaded {geojason_file_path}")
except Exception as e:
    print(f"Error loading GeoJSON file: {e}")
    gdf = None

if gdf is not None:
    print("\nFirst 5 rows of the GeoDataFrame:")
    display(gdf.head())

    print("\nInformation about the GeoDataFrame (columns, data types, non-null values):")
    gdf.info()

Successfully loaded /content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/train_labels/xView_train.geojson

First 5 rows of the GeoDataFrame:


,bounds_imcoords,edited_by,cat_id,type_id,ingest_time,index_right,image_id,point_geom,feature_id,grid_file,geometry
0,"2712,1145,2746,1177",wwoscarbecerril,1040010028371A00,73,2017-07-24 12:49:09.118000+00:00,2356,2355.tif,0101000020E6100000616E4E6406A256C03BE6ADA0D621...,374410,Grid2.shp,"POLYGON ((-90.5317 14.56604, -90.5317 14.56614..."
1,"2720,2233,2760,2288",wwoscarbecerril,1040010028371A00,73,2017-07-24 17:26:05.701000+00:00,2356,2355.tif,0101000020E6100000042D0CC705A256C0004F7071E71F...,394393,Grid2.shp,"POLYGON ((-90.53167 14.56222, -90.53167 14.562..."
2,"2687,1338,2740,1399",wwoscarbecerril,1040010028371A00,73,2017-07-24 12:45:09.081000+00:00,2356,2355.tif,0101000020E6100000B29F7E4707A256C0000AE6537921...,374031,Grid2.shp,"POLYGON ((-90.53179 14.56527, -90.53179 14.565..."
3,"2691,1201,2730,1268",wwoscarbecerril,1040010028371A00,73,2017-07-24 12:49:09.118000+00:00,2356,2355.tif,0101000020E6100000CE53137207A256C00EBF7084B521...,374409,Grid2.shp,"POLYGON ((-90.53177 14.56572, -90.53177 14.565..."
4,"2671,838,2714,869",wwoscarbecerril,1040010028371A00,73,2017-07-24 13:20:38.280000+00:00,2356,2355.tif,0101000020E610000060DE147608A256C05D6A384C6122...,377368,Grid2.shp,"POLYGON ((-90.53184 14.5671, -90.53184 14.5672..."



Information about the GeoDataFrame (columns, data types, non-null values):
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 601937 entries, 0 to 601936
Data columns (total 11 columns):
 #   Column           Non-Null Count   Dtype              
---  ------           --------------   -----              
 0   bounds_imcoords  601937 non-null  object             
 1   edited_by        601937 non-null  object             
 2   cat_id           601937 non-null  object             
 3   type_id          601937 non-null  int32              
 4   ingest_time      601937 non-null  datetime64[ms, UTC]
 5   index_right      601937 non-null  int32              
 6   image_id         601937 non-null  object             
 7   point_geom       601937 non-null  object             
 8   feature_id       601937 non-null  int32              
 9   grid_file        601937 non-null  object             
 10  geometry         601937 non-null  geometry           
dtypes: datetime64[ms, UTC](1), geomet

In [14]:
"""
Visualisation des annotations xView — version corrigée.
Utilise bounds_imcoords (coordonnées pixels) au lieu de geometry (lon/lat).
"""

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import geopandas as gpd

# --- Paramètres à adapter ---
image_directory = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/train_images"
geojson_path = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/train_labels/xView_train.geojson"
num_images_to_display = 10
min_labels_per_image = 1

gdf = gpd.read_file(geojson_path)
print(f"Total annotations : {len(gdf)}")

# Sélection des images les plus riches en annotations (pour être sûr de voir des boîtes)
counts = gdf['image_id'].value_counts()
counts = counts[counts >= min_labels_per_image]
selected_image_ids = counts.head(num_images_to_display).index.tolist()
print(counts.head(num_images_to_display))

class_names = {
    17: "Passenger Vehicle", 18: "Small Car", 19: "Bus", 20: "Pickup Truck",
    21: "Utility Truck", 23: "Truck", 24: "Cargo Truck", 25: "Truck w/Box",
    26: "Truck Tractor", 27: "Trailer", 28: "Truck w/Flatbed", 29: "Truck w/Liquid",
    32: "Crane Truck", 33: "Railway Vehicle", 34: "Passenger Car (Rail)",
    35: "Cargo Car (Rail)", 36: "Flat Car (Rail)", 37: "Tank Car (Rail)",
    38: "Locomotive", 40: "Maritime Vessel", 41: "Motorboat", 42: "Sailboat",
    44: "Tugboat", 45: "Barge", 47: "Fishing Vessel", 49: "Ferry",
    50: "Yacht", 51: "Container Ship", 52: "Oil Tanker", 53: "Engineering Vehicle",
    54: "Tower Crane", 55: "Container Crane", 56: "Reach Stacker", 57: "Straddle Carrier",
    59: "Mobile Crane", 60: "Dump Truck", 61: "Haul Truck", 62: "Scraper/Tractor",
    63: "Front Loader", 64: "Excavator", 65: "Cement Mixer", 66: "Ground Grader",
    71: "Hut/Tent", 72: "Shed", 73: "Building", 74: "Aircraft Hangar",
    76: "Damaged Building", 77: "Facility", 79: "Construction Site",
    83: "Vehicle Lot", 84: "Helipad", 86: "Storage Tank",
    89: "Shipping Container Lot", 91: "Shipping Container", 93: "Pylon", 94: "Tower"
}

def parse_bounds(bounds_str):
    xmin, ymin, xmax, ymax = map(float, bounds_str.split(','))
    return xmin, ymin, xmax, ymax

for img_id in selected_image_ids:
    image_path = os.path.join(image_directory, img_id)
    if not os.path.exists(image_path):
        print(f"Image introuvable : {image_path}, on passe.")
        continue

    try:
        img = Image.open(image_path)
        img_array = np.array(img)
    except Exception as e:
        print(f"Erreur chargement {image_path} : {e}")
        continue

    image_labels = gdf[gdf['image_id'] == img_id]
    print(f"\n{img_id} -> {len(image_labels)} objets annotés")

    fig, ax = plt.subplots(1, figsize=(14, 14))
    ax.imshow(img_array)
    ax.set_title(f"{img_id} — {len(image_labels)} objets détectés")
    ax.set_axis_off()

    for idx, row in image_labels.iterrows():
        xmin, ymin, xmax, ymax = parse_bounds(row['bounds_imcoords'])
        cls_id = row['type_id']
        label = class_names.get(cls_id, str(cls_id))

        rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                  linewidth=1.5, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        # ax.text(xmin, ymin - 5, label, color='yellow', fontsize=7,
        #         backgroundcolor='black', clip_on=True)

    plt.tight_layout()
    plt.show()
    plt.close(fig)

Output hidden; open in https://colab.research.google.com to view.

In [16]:
import requests

url = "https://raw.githubusercontent.com/DIUx-xView/data_utilities/master/xview_class_labels.txt"
response = requests.get(url)

class_names = {}
for line in response.text.strip().split('\n'):
    idx, name = line.split(':')
    class_names[int(idx)] = name.strip()

print(len(class_names))

60
